<a href="https://colab.research.google.com/github/gauravd12345/miniCLIP/blob/main/miniCLIP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("eeshawn/flickr30k")

print("Path to dataset files:", path)

images = path + '/flickr30k_images'
captions = path + '/captions.txt'

Using Colab cache for faster access to the 'flickr30k' dataset.
Path to dataset files: /kaggle/input/flickr30k


In [8]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F

from tqdm import tqdm

from torchvision import transforms
from PIL import Image
from IPython.display import display

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

device: cuda


In [9]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.add_special_tokens({"bos_token": "<sos>", "eos_token": "<eos>", "pad_token": "<pad>"})

3

In [10]:
vocab_len = len(tokenizer)
chunk_size = 64      # max seq len for text transformer

batch_size = 64
epochs = 10
lr = 2.5e-4

""" Vision Transformer parameters """
H = 128               # image height
W = 128               # image width
C = 3                 # number of channels

P = 16                # patch resolution
N_p = (H * W) // P**2   # number of patches

""" Transformer parameters """
d_model = 512
d_k = 64
d_v = 64
h = 8
N = 6

""" CLIP parameters """
d_i = 512
d_t = 512
d_e = 512

In [11]:
import pandas as pd

df = pd.read_csv(captions)

image_names = df['image_name']
comment_numbers = df['comment_number']
comments = df['comment']

In [12]:
class ViTDataset(Dataset):
    def __init__(self, image_names, captions, transform=None):
        self.captions = captions
        self.image_names = image_names
        self.transform = transform

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img = Image.open(f"{images}/{self.image_names[idx]}").convert("RGB")
        if self.transform:
            img = self.transform(img)
        img = img.reshape(N_p, P**2 * C)

        tokens = tokenizer(f"<sos> {self.captions[idx]} <eos>",
                          return_tensors="pt",
                          padding="max_length",
                          truncation=True,
                          max_length=chunk_size
                        )
        input_ids = tokens["input_ids"].squeeze(0)
        attention_mask = tokens["attention_mask"].squeeze(0)   # <-- add this

        return img, input_ids, attention_mask                  # <-- return mask


transform = transforms.Compose([
    transforms.Resize((H, W)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # <-- add this
])

dataset = ViTDataset(image_names, comments, transform=transform)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=2)

for img, caption, mask in dataloader:                      # <-- unpack mask
    print(f"Samples per batch: {len(dataloader)}")
    print(f"Batched images shape: {img.shape}")
    print(f"Batched captions shape: {caption.shape}")
    break


Samples per batch: 2484
Batched images shape: torch.Size([64, 64, 768])
Batched captions shape: torch.Size([64, 64])


In [13]:
import math

# training scheduler
total_steps = epochs * len(dataloader)
warmup_steps = int(0.05 * total_steps)

def lr_lambda(step): # cosine schedule
    if step < warmup_steps:
        return step / warmup_steps
    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

In [14]:
class Transformer(nn.Module):
  def __init__(self):
    super().__init__()

    self.embed = nn.Embedding(vocab_len, d_model)
    self.pos = nn.Embedding(chunk_size, d_model) # positional embedding

    self.W_q = nn.ModuleList([nn.ModuleList([nn.Linear(d_model, d_k) for _ in range(h)]) for _ in range(N)]) # q, k, v projections
    self.W_k = nn.ModuleList([nn.ModuleList([nn.Linear(d_model, d_k) for _ in range(h)]) for _ in range(N)])
    self.W_v = nn.ModuleList([nn.ModuleList([nn.Linear(d_model, d_v) for _ in range(h)]) for _ in range(N)])

    self.W_o = nn.ModuleList([nn.Linear(h * d_v, d_model) for _ in range(N)]) # final projection

    self.ffn1 = nn.ModuleList([nn.Linear(d_model, 4 * d_model) for _ in range(N)]) # ffn layer
    self.ffn2 = nn.ModuleList([nn.Linear(4 * d_model, d_model) for _ in range(N)])

    self.ln1 = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(N)])
    self.ln2 = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(N)])

    self.dropout = nn.Dropout(0.1)
    self.proj = nn.Linear(d_model, d_t)

  def multi_head_attention(self, x, layer_idx):
    W_tot = []
    for Q, K, V in zip(self.W_q[layer_idx], self.W_k[layer_idx], self.W_v[layer_idx]):
      Q_i = Q(x)
      K_i = K(x)
      V_i = V(x)

      alignment = torch.matmul(Q_i, K_i.transpose(-2, -1))                      # query-key alignment

      wei = self.dropout(torch.softmax(alignment / (d_k ** 0.5), dim=-1))       # alignment weights
      wei_value = torch.matmul(wei, V_i)                                        # weighted values

      W_tot.append(wei_value)

    out = self.W_o[layer_idx](torch.cat(W_tot, dim=2))
    return out

  def forward(self, x, mask=None):  # (batch_size, chunk_size)
    p = torch.arange(x.size(1)).to(x.device)
    x = self.dropout(self.embed(x) + self.pos(p))
    for i in range(N):
      normed = self.ln1[i](x)                        # Pre-LN
      out = self.multi_head_attention(normed, i)
      x = x + out

      normed = self.ln2[i](x)                        # Pre-LN
      fn = self.ffn2[i](F.gelu(self.ffn1[i](normed)))  # GELU
      x = x + self.dropout(fn)

    if mask is not None:
      mask = mask.unsqueeze(-1).float()              # (B, T, 1)
      out = (x * mask).sum(dim=1) / mask.sum(dim=1) # masked mean pool
    else:
      out = x.mean(dim=1)

    out = self.proj(out)                             # (B, d_t)
    return out


In [15]:
class VisionTransformer(nn.Module):
  def __init__(self):
    super().__init__()

    self.embed = nn.Linear(P**2 * C, d_model)
    self.pos = nn.Embedding(N_p, d_model) # positional embedding

    self.W_q = nn.ModuleList([nn.ModuleList([nn.Linear(d_model, d_k) for _ in range(h)]) for _ in range(N)]) # q, k, v projections
    self.W_k = nn.ModuleList([nn.ModuleList([nn.Linear(d_model, d_k) for _ in range(h)]) for _ in range(N)])
    self.W_v = nn.ModuleList([nn.ModuleList([nn.Linear(d_model, d_v) for _ in range(h)]) for _ in range(N)])

    self.W_o = nn.ModuleList([nn.Linear(h * d_v, d_model) for _ in range(N)]) # final projection

    self.ffn1 = nn.ModuleList([nn.Linear(d_model, 4 * d_model) for _ in range(N)]) # ffn layer
    self.ffn2 = nn.ModuleList([nn.Linear(4 * d_model, d_model) for _ in range(N)])

    self.ln1 = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(N)])
    self.ln2 = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(N)])

    self.dropout = nn.Dropout(0.1)
    self.proj = nn.Linear(d_model, d_i)

  def multi_head_attention(self, x, layer_idx):
    W_tot = []
    for Q, K, V in zip(self.W_q[layer_idx], self.W_k[layer_idx], self.W_v[layer_idx]):
      Q_i = Q(x)
      K_i = K(x)
      V_i = V(x)

      alignment = torch.matmul(Q_i, K_i.transpose(-2, -1))                      # query-key alignment

      wei = self.dropout(torch.softmax(alignment / (d_k ** 0.5), dim=-1))       # alignment weights
      wei_value = torch.matmul(wei, V_i)                                        # weighted values

      W_tot.append(wei_value)

    out = self.W_o[layer_idx](torch.cat(W_tot, dim=2))
    return out

  def forward(self, x):  # (B, N, P^2 * C)
    p = torch.arange(x.size(1)).to(x.device)
    x = self.dropout(self.embed(x) + self.pos(p))
    for i in range(N):
      normed = self.ln1[i](x)                           # Pre-LN
      out = self.multi_head_attention(normed, i)
      x = x + out

      normed = self.ln2[i](x)                           # Pre-LN
      fn = self.ffn2[i](F.gelu(self.ffn1[i](normed)))  # GELU
      x = x + self.dropout(fn)

    out = x.mean(dim=1)   # mean pool over patches — no padding here, all valid
    out = self.proj(out)  # (B, d_i)
    return out


In [16]:
class miniCLIP(nn.Module):
  def __init__(self):
    super().__init__()
    self.transformer = Transformer()
    self.vision_transformer = VisionTransformer()
    self.W_i = nn.Linear(d_i, d_e)
    self.W_t = nn.Linear(d_t, d_e)
    self.log_temperature = nn.Parameter(torch.log(torch.tensor(0.07)))

  def forward(self, img, caption, mask=None):
    I_f = self.vision_transformer(img)
    T_f = self.transformer(caption, mask=mask)

    I_e = F.normalize(self.W_i(I_f), p=2, dim=1)
    T_e = F.normalize(self.W_t(T_f), p=2, dim=1)

    temp = self.log_temperature.clamp(max=math.log(100)).exp()
    logits = torch.matmul(I_e, T_e.T) / temp
    return logits


In [17]:
model = miniCLIP().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

for epoch in range(epochs):
    total_loss = 0.0
    pbar = tqdm(dataloader, desc=f"Epoch {epoch + 1}/{epochs}")
    model.train()

    for img, caption, mask in pbar:
        optimizer.zero_grad()
        img, caption, mask = img.to(device), caption.to(device), mask.to(device)
        labels = torch.arange(img.size(0)).to(device)

        out = model(img, caption, mask=mask)
        loss_i = criterion(out, labels)
        loss_t = criterion(out.T, labels)
        loss = (loss_i + loss_t) / 2

        total_loss += loss.item()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        pbar.set_postfix(loss=f"{loss.item():.4f}")

    model.eval()
    with torch.no_grad():
        img_batch, cap_batch, mask_batch = next(iter(dataloader))
        img_batch, cap_batch, mask_batch = img_batch.to(device), cap_batch.to(device), mask_batch.to(device)
        logits = model(img_batch, cap_batch, mask=mask_batch)
        diag = logits.diag().mean().item()
        off_diag = (logits.sum() - logits.diag().sum()) / (batch_size * (batch_size - 1))

    print(f"Epoch: {epoch + 1}/{epochs} | Loss: {total_loss / len(dataloader):.4f} | Gap: {diag - off_diag.item():.4f}")

Epoch 1/10:   7%|▋         | 185/2484 [01:37<20:16,  1.89it/s, loss=4.1911]


KeyboardInterrupt: 